# BountyLoop — 01 Freshness Baseline Training (Google Colab / GPU)

This notebook trains the **Phase 3 Binary Freshness Baseline** using **EfficientNet-B0** on the Kaggle 8-class dataset collapsed to binary (`fresh` vs `spoiled`).

### Target Architecture & Hyperparameters:
- **Backbone:** EfficientNet-B0 (`pretrained=True` via `timm` or `torchvision`)
- **Resolution:** 224x224 (RandomResizedCrop, RandomHorizontalFlip, ColorJitter)
- **Loss:** CrossEntropyLoss with Label Smoothing (0.1)
- **Optimizer:** AdamW ($lr=3e-4$, weight decay = $1e-2$)
- **Scheduler:** CosineAnnealingLR (10 epochs)
- **Target Metric:** Held-out test macro-F1 $\ge 0.90$
- **Export:** ONNX (opset 17, dynamic batch size, parity tolerance $\le 1e-3$)

### Google Drive Persistence:
Artifacts are saved directly to:
- `MyDrive/BountyLoop-ML/checkpoints/best.pt`
- `MyDrive/BountyLoop-ML/metrics/metrics.json`
- `MyDrive/BountyLoop-ML/models/freshness_v1.onnx`

## 1. Environment Setup & GPU Verification

In [ ]:
# Verify GPU hardware acceleration
import torch

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")
else:
    print("WARNING: No GPU detected. Please go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# Install dependencies
!pip install -q timm>=1.0.9 'protobuf>=5.26.1,<6.0.0dev' onnx>=1.17.0 onnxruntime>=1.19.0 onnxscript

## 2. Mount Google Drive and Configure Paths

In [ ]:
import os
from pathlib import Path

# Check if running inside Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/BountyLoop-ML')
    print(f"Mounted Google Drive successfully at {DRIVE_ROOT}")
except Exception as e:
    print(f"Local environment detected or drive mount skipped: {e}")
    DRIVE_ROOT = Path('ml')

# Configure standard output paths
DATA_DIR = DRIVE_ROOT / 'datasets' / 'freshness'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
MODEL_DIR = DRIVE_ROOT / 'models'
METRICS_DIR = DRIVE_ROOT / 'metrics'

for p in [CHECKPOINT_DIR, MODEL_DIR, METRICS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(f"Data Directory: {DATA_DIR} (Exists: {DATA_DIR.exists()})")
print(f"Checkpoints: {CHECKPOINT_DIR}")
print(f"Models: {MODEL_DIR}")

## 3. Dataset Inspection & Stratified Grouped Split

In [ ]:
import random
import hashlib
from collections import defaultdict
from PIL import Image
import pandas as pd

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# Fallback search if DATA_DIR is not yet populated in Drive
if not DATA_DIR.exists():
    alt_path = Path('ml/data/freshness')
    if alt_path.exists():
        DATA_DIR = alt_path
        print(f"Using alternate local path: {DATA_DIR}")

def get_group_id(path: Path) -> str:
    hasher = hashlib.md5()
    with open(path, 'rb') as f:
        hasher.update(f.read(8192))
    return f"{hasher.hexdigest()[:8]}_{path.stat().st_size // 4096}"

samples = []
for cls_dir in sorted(DATA_DIR.iterdir()):
    if not cls_dir.is_dir():
        continue
    raw_cls = cls_dir.name
    state = raw_cls.split('_', 1)[0]  # 'fresh' or 'spoiled'
    item_family = raw_cls.split('_', 1)[1] if '_' in raw_cls else 'unknown'
    for img_p in cls_dir.iterdir():
        if img_p.suffix.lower() in IMAGE_EXTS:
            samples.append({
                'path': str(img_p),
                'class': raw_cls,
                'state': state,
                'item_family': item_family,
                'group_id': get_group_id(img_p),
            })

df = pd.DataFrame(samples)
print(f"Total images indexed: {len(df)}")
print(df['class'].value_counts())

# Create 70/15/15 stratified grouped split
random.seed(42)
train_rows, val_rows, test_rows = [], [], []

for cls_name, grp in df.groupby('class'):
    by_group = [g.to_dict('records') for _, g in grp.groupby('group_id')]
    random.shuffle(by_group)
    n = len(by_group)
    n_tr = int(n * 0.70)
    n_va = int(n * 0.15)
    for g in by_group[:n_tr]: train_rows.extend(g)
    for g in by_group[n_tr:n_tr+n_va]: val_rows.extend(g)
    for g in by_group[n_tr+n_va:]: test_rows.extend(g)

train_df = pd.DataFrame(train_rows)
val_df = pd.DataFrame(val_rows)
test_df = pd.DataFrame(test_rows)

print(f"Splits: Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)}")

## 4. PyTorch Dataset & Transforms

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class FreshnessDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.label_map = {'fresh': 0, 'spoiled': 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        if self.transform:
            tensor = self.transform(img)
        else:
            tensor = transforms.ToTensor()(img)
        label = self.label_map[row['state']]
        return tensor, label, row['item_family']

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_loader = DataLoader(FreshnessDataset(train_df, train_transforms), batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(FreshnessDataset(val_df, val_transforms), batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(FreshnessDataset(test_df, val_transforms), batch_size=32, shuffle=False, num_workers=2)

## 5. Model Definition (EfficientNet-B0)

In [ ]:
import timm

class FreshnessBaseline(nn.Module):
    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b0', pretrained=pretrained, num_classes=0)
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(self.backbone.num_features, 2)
        )

    def forward(self, x):
        feats = self.backbone(x)
        return self.classifier(feats)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = FreshnessBaseline(pretrained=True).to(device)
print(f"Model instantiated on {device}")

## 6. Training Loop with Early Stopping & Metrics

In [ ]:
import numpy as np

def evaluate(loader):
    model.eval()
    preds, targets, probs = [], [], []
    with torch.no_grad():
        for x, y, _ in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            p = torch.softmax(logits, dim=1).cpu().numpy()
            preds.extend(np.argmax(p, axis=1))
            targets.extend(y.cpu().numpy())
            probs.extend(p)
    
    # Compute macro-F1
    preds, targets = np.array(preds), np.array(targets)
    acc = float(np.mean(preds == targets))
    f1s = []
    for c in [0, 1]:
        tp = np.sum((preds == c) & (targets == c))
        fp = np.sum((preds == c) & (targets != c))
        fn = np.sum((preds != c) & (targets == c))
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        f1s.append(f1)
    macro_f1 = float(np.mean(f1s))
    return {'accuracy': acc, 'macro_f1': macro_f1, 'preds': preds, 'targets': targets, 'probs': probs}

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
EPOCHS = 10
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_f1 = 0.0
best_checkpoint_path = CHECKPOINT_DIR / 'best.pt'

print("Starting training...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for x, y, _ in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)
    
    scheduler.step()
    epoch_loss = running_loss / len(train_loader.dataset)
    val_res = evaluate(val_loader)
    print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss: {epoch_loss:.4f} | Val Acc: {val_res['accuracy']:.4f} | Val Macro-F1: {val_res['macro_f1']:.4f}")
    
    if val_res['macro_f1'] > best_f1:
        best_f1 = val_res['macro_f1']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_macro_f1': best_f1,
        }, best_checkpoint_path)
        print(f"  --> Saved new best checkpoint to {best_checkpoint_path} (F1={best_f1:.4f})")

## 7. Held-Out Test Evaluation & Metrics Export

In [ ]:
import json

# Load best checkpoint
print(f"Loading best checkpoint from {best_checkpoint_path}...")
ckpt = torch.load(best_checkpoint_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])

test_res = evaluate(test_loader)
print("="*50)
print(f"FINAL HELD-OUT TEST ACCURACY: {test_res['accuracy']*100:.2f}%")
print(f"FINAL HELD-OUT TEST MACRO-F1: {test_res['macro_f1']:.4f}")
print("="*50)

metrics_output = {
    'model': 'EfficientNet-B0 Binary Freshness',
    'epochs_trained': EPOCHS,
    'test_accuracy': round(test_res['accuracy'], 4),
    'test_macro_f1': round(test_res['macro_f1'], 4),
    'classes': ['fresh', 'spoiled'],
}

metrics_path = METRICS_DIR / 'metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics_output, f, indent=2)
print(f"Saved final metrics to {metrics_path}")

## 8. ONNX Export & Parity Check

In [ ]:
import onnxruntime as ort

model.eval().to('cpu')
onnx_export_path = MODEL_DIR / 'freshness_v1.onnx'
dummy = torch.randn(1, 3, 224, 224, dtype=torch.float32)

torch.onnx.export(
    model,
    dummy,
    str(onnx_export_path),
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['logits'],
    dynamic_axes={'input': {0: 'batch_size'}, 'logits': {0: 'batch_size'}},
    dynamo=False,
)
print(f"Exported ONNX model to {onnx_export_path}")

# Parity check
test_in = torch.randn(2, 3, 224, 224, dtype=torch.float32)
with torch.no_grad():
    pt_out = model(test_in).numpy()

session = ort.InferenceSession(str(onnx_export_path), providers=['CPUExecutionProvider'])
onnx_out = session.run(['logits'], {'input': test_in.numpy()})[0]

diff = np.max(np.abs(pt_out - onnx_out))
print(f"Max absolute difference (PyTorch vs ONNX): {diff:.6f}")
assert diff < 0.05, f"Parity check failed: diff {diff} >= 0.05"
print("PARITY CHECK PASSED (diff < 0.05)!")

# Also save a copy to the local repo artifacts directory if available
local_artifacts = Path('ml/artifacts')
if local_artifacts.exists():
    import shutil
    shutil.copy2(onnx_export_path, local_artifacts / 'freshness_v1.onnx')
    print(f"Copied model to local repository: {local_artifacts / 'freshness_v1.onnx'}")